In [ ]:
import shutil
import os

src = "/kaggle/input/ao-data"
dst = "/kaggle/working/ao_data"
if not os.path.exists(dst):
    shutil.copytree(src, dst)

In [ ]:
!pip install timm opencv-python scikit-learn matplotlib torchmetrics tqdm

In [ ]:
#Data augmentation and split with filestem
import os
import cv2
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
from torchvision import transforms

# ============================
# AUGMENTATION TRANSFORMS
# ============================
base_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.ToPILImage()
])

aug_transform = transforms.Compose([
    transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
])

# ============================
# DIRECTORIES
# ============================
root_dir = "/kaggle/working/ao_data/ao_crops"
out_train = "/kaggle/working/ao_data/balanced_train"
out_val = "/kaggle/working/ao_data/balanced_val"
out_test = "/kaggle/working/ao_data/balanced_test"

for d in [out_train, out_val, out_test]:
    os.makedirs(d, exist_ok=True)

# Target sizes
N_TRAIN, N_VAL, N_TEST = 800, 100, 100

def ensure_class_dir(base, cls):
    out = os.path.join(base, cls)
    os.makedirs(out, exist_ok=True)
    return out

# ============================
# BALANCE & SPLIT WITH ORIGINAL FILENAMES
# ============================
for cls in tqdm(os.listdir(root_dir)):
    cls_src = os.path.join(root_dir, cls)
    imgs = [os.path.join(cls_src, f) for f in os.listdir(cls_src) if f.endswith(".png")]
    random.shuffle(imgs)
    
    count_raw = len(imgs)
    n_split = min(count_raw, N_TRAIN + N_VAL + N_TEST)
    imgs_use = imgs[:n_split]
    
    # Assign as much as possible from originals
    raw_train = imgs_use[:min(count_raw, N_TRAIN)]
    raw_val = imgs_use[min(N_TRAIN, count_raw):min(N_TRAIN + N_VAL, count_raw)]
    raw_test = imgs_use[min(N_TRAIN + N_VAL, count_raw):min(N_TRAIN + N_VAL + N_TEST, count_raw)]
    
    # ====================
    # TRAIN SET
    # ====================
    out_dir = ensure_class_dir(out_train, cls)
    
    # Save original images with their original filenames
    for img_path in raw_train:
        img = Image.open(img_path).convert('RGB')
        img_aug = base_transform(img)
        
        # Keep original filename
        original_filename = os.path.basename(img_path)
        img_aug.save(os.path.join(out_dir, original_filename))
    
    # Generate augmented images if needed
    needed = N_TRAIN - len(raw_train)
    for idx in range(needed):
        img_path = random.choice(raw_train)
        img = Image.open(img_path).convert('RGB')
        img_aug = aug_transform(img)
        img_aug = base_transform(img_aug)
        
        # Extract original filename without extension
        original_filename = os.path.basename(img_path)
        base_name = os.path.splitext(original_filename)[0]
        
        # Save as: originalname_aug_0.png, originalname_aug_1.png, etc.
        aug_filename = f"{base_name}_aug_{idx}.png"
        img_aug.save(os.path.join(out_dir, aug_filename))
    
    # ====================
    # VALIDATION SET
    # ====================
    out_dir = ensure_class_dir(out_val, cls)
    
    # Save original images with their original filenames
    for img_path in raw_val:
        img = Image.open(img_path).convert('RGB')
        img_aug = base_transform(img)
        
        # Keep original filename
        original_filename = os.path.basename(img_path)
        img_aug.save(os.path.join(out_dir, original_filename))
    
    # Generate augmented images if needed
    needed = N_VAL - len(raw_val)
    for idx in range(needed):
        source_pool = raw_val if raw_val else raw_train
        img_path = random.choice(source_pool)
        img = Image.open(img_path).convert('RGB')
        img_aug = aug_transform(img)
        img_aug = base_transform(img_aug)
        
        # Extract original filename without extension
        original_filename = os.path.basename(img_path)
        base_name = os.path.splitext(original_filename)[0]
        
        # Save as: originalname_aug_0.png, originalname_aug_1.png, etc.
        aug_filename = f"{base_name}_aug_{idx}.png"
        img_aug.save(os.path.join(out_dir, aug_filename))
    
    # ====================
    # TEST SET
    # ====================
    out_dir = ensure_class_dir(out_test, cls)
    
    # Save original images with their original filenames
    for img_path in raw_test:
        img = Image.open(img_path).convert('RGB')
        img_aug = base_transform(img)
        
        # Keep original filename
        original_filename = os.path.basename(img_path)
        img_aug.save(os.path.join(out_dir, original_filename))
    
    # Generate augmented images if needed
    needed = N_TEST - len(raw_test)
    for idx in range(needed):
        source_pool = raw_test if raw_test else raw_train
        img_path = random.choice(source_pool)
        img = Image.open(img_path).convert('RGB')
        img_aug = aug_transform(img)
        img_aug = base_transform(img_aug)
        
        # Extract original filename without extension
        original_filename = os.path.basename(img_path)
        base_name = os.path.splitext(original_filename)[0]
        
        # Save as: originalname_aug_0.png, originalname_aug_1.png, etc.
        aug_filename = f"{base_name}_aug_{idx}.png"
        img_aug.save(os.path.join(out_dir, aug_filename))

# ====================
# PRINT DISTRIBUTION
# ====================
print("\nTrain distribution:")
for cls in os.listdir(out_train):
    count = len([f for f in os.listdir(os.path.join(out_train, cls)) if f.endswith('.png')])
    print(f" {cls}: {count}")

print("\nVal distribution:")
for cls in os.listdir(out_val):
    count = len([f for f in os.listdir(os.path.join(out_val, cls)) if f.endswith('.png')])
    print(f" {cls}: {count}")

print("\nTest distribution:")
for cls in os.listdir(out_test):
    count = len([f for f in os.listdir(os.path.join(out_test, cls)) if f.endswith('.png')])
    print(f" {cls}: {count}")


In [ ]:
# ==========================
# AGE-INTEGRATED CONVNEXT MODEL
# Age Range: 0-17 years (Pediatric)
# ==========================
import os
import cv2
import random
import re
import numpy as np
from PIL import Image
from tqdm import tqdm
from collections import Counter
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_recall_curve, auc, roc_curve, average_precision_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm
import matplotlib.pyplot as plt

# ==========================
# CONFIGURATION - BALANCED FOR PERFORMANCE & AGE SENSITIVITY
# ==========================
CONFIG = {
    'img_size': 256,
    'batch_size': 32,
    'num_epochs': 40,  
    'learning_rate': 0.0001,
    'weight_decay': 0.005,
    'patience': 10,  
    'n_train': 800,
    'n_val': 100,
    'n_test': 100,
    'age_embed_dim': 16,
    'age_loss_weight': 0.02,  
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================
# CLAHE Transform
# ==========================
class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size
    
    def __call__(self, img):
        img_np = np.array(img)
        if len(img_np.shape) == 2:
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
            img_np = clahe.apply(img_np)
        else:
            lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
            lab[:, :, 0] = clahe.apply(lab[:, :, 0])
            img_np = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        return Image.fromarray(img_np)

# ==========================
# AGE EXTRACTION FUNCTION
# ==========================
def extract_age_from_filename(filename):
    """
    Extract age from filename pattern: _M013_ or _F025_ or -F019-, etc.
    Returns age as integer, or -1 if not found
    Age range: 0-17 (pediatric)
    """
    match = re.search(r'[_\-][MFmf](\d{3})[_\-]', filename)
    if match:
        age = int(match.group(1))
        return age
    return -1  # Unknown age

# ==========================
# DATASET WITH AGE
# ==========================
class AgeAwareImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.class_to_idx = {}
        
        # Build dataset
        classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        
        for cls in classes:
            cls_dir = os.path.join(root_dir, cls)
            for filename in os.listdir(cls_dir):
                if filename.endswith('.png'):
                    img_path = os.path.join(cls_dir, filename)
                    age = extract_age_from_filename(filename)
                    if age >= 0:  # Only include samples with valid age
                        self.samples.append((img_path, self.class_to_idx[cls], age))
        
        self.classes = classes
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label, age = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        
        if self.transform:
            img = self.transform(img)
        
        # Normalize age for pediatric range (0-17 years)
        age_normalized = age / 17.0 if (0 <= age <= 17) else 0.0
        
        return img, label, torch.tensor(age_normalized, dtype=torch.float32)

# ==========================
# DATA AUGMENTATION
# ==========================
train_transform = transforms.Compose([
    CLAHETransform(),
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(0.3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    CLAHETransform(),
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================
# LOAD DATASETS
# ==========================
out_train = "/kaggle/working/ao_data/balanced_train"
out_val = "/kaggle/working/ao_data/balanced_val"
out_test = "/kaggle/working/ao_data/balanced_test"

train_dataset = AgeAwareImageDataset(out_train, transform=train_transform)
val_dataset = AgeAwareImageDataset(out_val, transform=val_test_transform)
test_dataset = AgeAwareImageDataset(out_test, transform=val_test_transform)

class_names = train_dataset.classes
n_classes = len(class_names)

print(f"Classes: {class_names}")
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")

# ==========================
# COMPUTE CLASS WEIGHTS
# ==========================
train_labels = [label for _, label, _ in train_dataset.samples]
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print(f"\nClass weights: {class_weights}")

# ==========================
# DATA LOADERS
# ==========================
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

# ==========================
# BALANCED AGE-AWARE CONVNEXT MODEL
# ==========================
class AgeAwareConvNeXt(nn.Module):
    def __init__(self, num_classes, age_embed_dim=32):
        super(AgeAwareConvNeXt, self).__init__()
        
        # Load pretrained ConvNeXt Small
        self.backbone = timm.create_model('convnext_small', pretrained=True, num_classes=0)
        backbone_out_dim = self.backbone.num_features  # 768 for convnext_small
        
        # Age embedding network
        self.age_embed = nn.Sequential(
            nn.Linear(1, age_embed_dim),
            nn.ReLU(),
            nn.BatchNorm1d(age_embed_dim),
            nn.Dropout(0.1),  
            nn.Linear(age_embed_dim, age_embed_dim),
            nn.ReLU(),
            nn.BatchNorm1d(age_embed_dim),
            nn.Dropout(0.1)   
        )
        
        
        self.age_predictor = nn.Sequential(
            nn.Linear(age_embed_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
        # Combined classifier
        combined_dim = backbone_out_dim + age_embed_dim
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),  
            nn.Linear(combined_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),  
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.1),  
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x, age, return_age_pred=False):
        # Extract image features
        img_features = self.backbone(x)  # [batch_size, 768]
        
        # Process age
        age = age.unsqueeze(1)  # [batch_size, 1]
        age_features = self.age_embed(age)  # [batch_size, age_embed_dim]
        
        age_pred = self.age_predictor(age_features).squeeze(1)  # [batch_size]
        
        # Concatenate image and age features
        combined = torch.cat([img_features, age_features], dim=1)
        
        # Final classification
        output = self.classifier(combined)
        
        if return_age_pred:
            return output, age_pred
        return output

# ==========================
# FOCAL LOSS
# ==========================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.weight = weight
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# ==========================
# MODEL SETUP
# ==========================
model = AgeAwareConvNeXt(num_classes=n_classes, age_embed_dim=CONFIG['age_embed_dim'])
model = model.to(device)

criterion = FocalLoss(gamma=2.0, weight=class_weights_tensor)
age_criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# ==========================
# TRAINING WITH BALANCED AGE LEARNING
# ==========================
from torch.cuda.amp import GradScaler, autocast

scaler = GradScaler()

# History tracking
train_acc_history = []
val_acc_history = []
train_loss_history = []
val_loss_history = []
precision_history = []
recall_history = []
f1_history = []
age_loss_history = []

best_val_acc = 0.0
best_val_loss = float('inf')
patience_counter = 0

print("\n" + "="*80)
print("TRAINING BALANCED AGE-INTEGRATED CONVNEXT MODEL")
print("Age Range: 0-17 years (Pediatric)")
print(f"Age Loss Weight: {CONFIG['age_loss_weight']}")
print("="*80)

for epoch in range(CONFIG['num_epochs']):
    model.train()
    running_loss = 0.0
    running_cls_loss = 0.0
    running_age_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels, ages in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']}"):
        images, labels, ages = images.to(device), labels.to(device), ages.to(device)
        
        optimizer.zero_grad()
        
        with autocast():
            outputs, age_pred = model(images, ages, return_age_pred=True)
            
            # Classification loss
            cls_loss = criterion(outputs, labels)
            
            # Age prediction loss (auxiliary task with reduced weight)
            age_loss = age_criterion(age_pred, ages)
            
            # Combined loss with reduced age contribution
            loss = cls_loss + CONFIG['age_loss_weight'] * age_loss
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        running_cls_loss += cls_loss.item()
        running_age_loss += age_loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_acc = 100 * correct / total
    avg_loss = running_loss / len(train_loader)
    avg_cls_loss = running_cls_loss / len(train_loader)
    avg_age_loss = running_age_loss / len(train_loader)
    train_acc_history.append(train_acc)
    train_loss_history.append(avg_loss)
    age_loss_history.append(avg_age_loss)
    
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    val_running_loss = 0.0
    all_val_preds = []
    all_val_labels = []
    
    with torch.no_grad():
        for images, labels, ages in val_loader:
            images, labels, ages = images.to(device), labels.to(device), ages.to(device)
            outputs, age_pred = model(images, ages, return_age_pred=True)
            
            cls_loss = criterion(outputs, labels)
            age_loss = age_criterion(age_pred, ages)
            loss = cls_loss + CONFIG['age_loss_weight'] * age_loss
            
            val_running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())
    
    val_acc = 100 * val_correct / val_total
    val_loss = val_running_loss / len(val_loader)
    val_acc_history.append(val_acc)
    val_loss_history.append(val_loss)
    
    # Compute macro precision, recall, F1
    precision_epoch = precision_score(all_val_labels, all_val_preds, average='macro', zero_division=0)
    recall_epoch = recall_score(all_val_labels, all_val_preds, average='macro', zero_division=0)
    f1_epoch = f1_score(all_val_labels, all_val_preds, average='macro', zero_division=0)
    
    precision_history.append(precision_epoch)
    recall_history.append(recall_epoch)
    f1_history.append(f1_epoch)
    
    print(f"Epoch {epoch+1}/{CONFIG['num_epochs']} | Train Acc: {train_acc:.2f}% | Train Loss: {avg_loss:.4f} | Age Loss: {avg_age_loss:.4f} | Val Acc: {val_acc:.2f}% | Val Loss: {val_loss:.4f} | P: {precision_epoch:.4f} | R: {recall_epoch:.4f} | F1: {f1_epoch:.4f}")
    
    scheduler.step()
    
    # Early stopping - prioritize accuracy but also track loss
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/kaggle/working/best_age_convnext.pth")
        print(f"Model saved! Val Acc: {val_acc:.2f}%, Val Loss: {val_loss:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

print(f"Training complete! Best Val Acc: {best_val_acc:.2f}%, Best Val Loss: {best_val_loss:.4f}")

# ==========================
# AGE EMBEDDING ANALYSIS
# ==========================
print("\n" + "="*80)
print("AGE EMBEDDING ANALYSIS")
print("="*80)

model.load_state_dict(torch.load("/kaggle/working/best_age_aware_convnext.pth"))
model.eval()

# Check age embedding weights
age_embed_first_layer = model.age_embed[0].weight.data
print(f"Age embedding first layer weight stats:")
print(f"  Mean: {age_embed_first_layer.mean():.6f}")
print(f"  Std: {age_embed_first_layer.std():.6f}")
print(f"  Min: {age_embed_first_layer.min():.6f}")
print(f"  Max: {age_embed_first_layer.max():.6f}")

# Test with different ages (0-17 years)
test_ages = [0, 4, 9, 13, 17]
print(f"\nAge embedding outputs for different ages (0-17 years):")
with torch.no_grad():
    for age in test_ages:
        age_norm = torch.tensor([[age/17.0]], dtype=torch.float32).to(device)
        age_feat = model.age_embed(age_norm)
        print(f"  Age {age}: mean={age_feat.mean():.4f}, std={age_feat.std():.4f}, L2 norm={torch.norm(age_feat):.4f}")

# ==========================
# TESTING
# ==========================
model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels, ages in test_loader:
        images, labels, ages = images.to(device), labels.to(device), ages.to(device)
        outputs = model(images, ages)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

test_acc = 100 * np.sum(all_labels == all_preds) / len(all_labels)
print(f"\nTest Accuracy: {test_acc:.2f}%")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# ==========================
# mAP AND ROC CALCULATION
# ==========================
all_labels_bin = label_binarize(all_labels, classes=list(range(n_classes)))

# Calculate mAP
map_per_class = average_precision_score(all_labels_bin, all_probs, average=None)
map_overall = average_precision_score(all_labels_bin, all_probs, average='macro')

print(f"\nmAP (Mean Average Precision) Metrics:")
for i, class_name in enumerate(class_names):
    print(f"  {class_name}: {map_per_class[i]:.4f}")
print(f"Overall mAP: {map_overall:.4f}")

# Calculate ROC AUC
roc_auc_per_class = roc_auc_score(all_labels_bin, all_probs, average=None)
roc_auc_overall = roc_auc_score(all_labels_bin, all_probs, average='macro')

print(f"\nROC AUC Metrics:")
for i, class_name in enumerate(class_names):
    print(f"  {class_name}: {roc_auc_per_class[i]:.4f}")
print(f"Overall ROC AUC: {roc_auc_overall:.4f}")

# ==========================
# VISUALIZATION
# ==========================

# 1. Train vs Val Accuracy
epochs_range = range(1, len(train_acc_history) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs_range, train_acc_history, label='Train Accuracy', marker='o', linewidth=2, color='#3498db')
plt.plot(epochs_range, val_acc_history, label='Validation Accuracy', marker='s', linewidth=2, color='#e74c3c')
plt.fill_between(epochs_range, train_acc_history, val_acc_history, alpha=0.2, color='gray', label='Train-Val Gap')
plt.xlabel('Epoch', fontsize=14)
plt.ylabel('Accuracy (%)', fontsize=14)
plt.title('Train vs Validation Accuracy - Balanced Age-Aware ConvNeXt', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/results/age_aware_train_val_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Train vs Val Loss
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, train_loss_history, label='Train Loss', marker='o', linewidth=2, color='#3498db')
plt.plot(epochs_range, val_loss_history, label='Validation Loss', marker='s', linewidth=2, color='#e74c3c')
plt.xlabel('Epoch', fontsize=14)
plt.ylabel('Loss', fontsize=14)
plt.title('Train vs Validation Loss - Balanced Age-Aware ConvNeXt', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/results/age_aware_train_val_loss.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Precision, Recall, F1
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, precision_history, label='Macro Precision', marker='o', linewidth=2)
plt.plot(epochs_range, recall_history, label='Macro Recall', marker='s', linewidth=2)
plt.plot(epochs_range, f1_history, label='Macro F1-Score', marker='^', linewidth=2)
plt.xlabel('Epoch', fontsize=14)
plt.ylabel('Score', fontsize=14)
plt.title('Precision, Recall, F1 During Training', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/results/age_aware_prec_recall_training.png', dpi=300, bbox_inches='tight')
plt.show()

# 4. Age Loss
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, age_loss_history, label='Age Prediction Loss', marker='o', linewidth=2, color='purple')
plt.xlabel('Epoch', fontsize=14)
plt.ylabel('MSE Loss', fontsize=14)
plt.title(f'Age Prediction Loss During Training (weight={CONFIG["age_loss_weight"]})', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/results/age_prediction_loss.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. ROC Curves
plt.figure(figsize=(12, 8))
colors = plt.cm.get_cmap('tab10', n_classes)

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC={roc_auc_per_class[i]:.3f})', 
             color=colors(i), linewidth=2)

fpr_macro, tpr_macro, _ = roc_curve(all_labels_bin.ravel(), all_probs.ravel())
plt.plot(fpr_macro, tpr_macro, label=f'Macro-average (AUC={roc_auc_overall:.3f})', 
         color='black', linestyle='--', linewidth=3)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random Classifier')

plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.title('ROC Curves - Balanced Age-Aware ConvNeXt (Test Set)', fontsize=16)
plt.legend(fontsize=10, loc='lower right')
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/results/age_aware_test_roc.png', dpi=300, bbox_inches='tight')
plt.show()

# 6. Train vs Test Accuracy
plt.figure(figsize=(8, 6))
final_train_acc = train_acc_history[-1]
bar_colors = ['#3498db', '#e74c3c']
bars = plt.bar(['Train', 'Test'], [final_train_acc, test_acc], color=bar_colors, alpha=0.8, edgecolor='black', linewidth=1.5)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}%',
             ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.ylabel('Accuracy (%)', fontsize=14)
plt.title('Train vs Test Accuracy - Balanced Age-Aware ConvNeXt', fontsize=16)
plt.ylim(0, 105)
plt.grid(True, axis='y', alpha=0.3)
plt.savefig('/kaggle/working/results/age_aware_train_vs_test_acc.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================
# FINAL SUMMARY
# ==========================
print(f"\n{'='*80}")
print("FINAL RESULTS - BALANCED AGE-INTEGRATED CONVNEXT")
print(f"{'='*80}")
print(f"Final Training Accuracy: {final_train_acc:.2f}%")
print(f"Final Test Accuracy: {test_acc:.2f}%")
print(f"Final Precision: {precision_history[-1]:.4f}")
print(f"Final Recall: {recall_history[-1]:.4f}")
print(f"Final F1-Score: {f1_history[-1]:.4f}")
print(f"Train-Val Gap: {abs(train_acc_history[-1] - val_acc_history[-1]):.2f}%")
print(f"Train-Test Gap: {abs(final_train_acc - test_acc):.2f}%")
print(f"Overall mAP: {map_overall:.4f}")
print(f"Overall ROC AUC: {roc_auc_overall:.4f}")
print(f"Final Age Loss: {age_loss_history[-1]:.4f}")
print(f"Age Loss Weight: {CONFIG['age_loss_weight']}")
print(f"{'='*80}")


In [ ]:
# EXPLAINABILITY FOR AGE-AWARE CONVNEXT
# =====================================================================
!pip install grad-cam captum shap

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, XGradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import shap
import os

# Create output directory
os.makedirs('/kaggle/working/gradcam_samples/', exist_ok=True)

# =====================================================================
# LOAD TRAINED MODEL WEIGHTS
# =====================================================================
print("Loading trained model weights...")
model.load_state_dict(torch.load("/kaggle/working/best_age_aware_convnext.pth"))

model.eval()
print("Loaded trained model weights from /kaggle/working/best_age_aware_convnext.pth")

# =====================================================================
# 1. GRAD-CAM VISUALIZATION (Visual Explainability)
# =====================================================================

class ConvNeXtGradCAM:
    """GradCAM for Age-Aware ConvNeXt Small"""
    
    def __init__(self, model, target_layer, cam_cls):
        self.model = model
        self.target_layer = target_layer
        self.cam_cls = cam_cls
    
    def generate_cam(self, input_tensor, age_tensor, target_class=None):
        """
        Generate GradCAM heatmap
        
        Args:
            input_tensor: preprocessed image tensor [1, 3, H, W]
            age_tensor: age value tensor [1]
            target_class: target class index (if None, uses predicted class)
        """
        # Forward pass to get prediction
        self.model.eval()
        with torch.no_grad():
            output = self.model(input_tensor, age_tensor)
            pred_class = output.argmax(dim=1).item()
        
        if target_class is None:
            target_class = pred_class
        
        # Define target for GradCAM
        targets = [ClassifierOutputTarget(target_class)]
        
        # Create a wrapper model that handles age input
        class AgeAwareWrapper(torch.nn.Module):
            def __init__(self, model, age_tensor):
                super().__init__()
                self.model = model
                self.age_tensor = age_tensor
            
            def forward(self, x):
                # Expand age_tensor to match batch size
                batch_size = x.shape[0]
                age_batch = self.age_tensor.expand(batch_size)
                return self.model(x, age_batch)
        
        # Create wrapper
        wrapped_model = AgeAwareWrapper(self.model, age_tensor) 
        
        cam_method = self.cam_cls(model =wrapped_model, target_layers =[self.target_layer])
        
        
        
        # Get GradCAM
        grayscale_cam = cam_method(
            input_tensor=input_tensor,
            targets=targets,
            eigen_smooth=True,
            aug_smooth=True
        )
        
        grayscale_cam = grayscale_cam[0, :]
        return grayscale_cam, pred_class, target_class
    
    def visualize(self, image_path, age, class_names, save_path=None):
        """
        Complete visualization pipeline
    
        Args:
            image_path: path to input image
            age: patient age (0-19 years, pediatric)
            class_names: list of class names
            save_path: path to save visualization
        """
        # Load and preprocess image
        img = Image.open(image_path).convert('RGB')
        img_np = np.array(img)
    
        # Apply same transforms as validation
        img_tensor = val_test_transform(img).unsqueeze(0).to(device)
        age_normalized = age / 19.0  # Normalize to [0, 1] for pediatric range
        age_tensor = torch.tensor([age_normalized], dtype=torch.float32).to(device)
    
        # Generate GradCAM
        cam, pred_class, target_class = self.generate_cam(img_tensor, age_tensor)
    
        # Prepare image for visualization
        img_resized = cv2.resize(img_np, (CONFIG['img_size'], CONFIG['img_size']))
        img_normalized = img_resized.astype(np.float32) / 255.0
    
        # Resize CAM to match img_normalized dimensions
        cam_resized = cv2.resize(cam, (img_normalized.shape[1], img_normalized.shape[0]))
    
        # Overlay CAM on image
        visualization = show_cam_on_image(img_normalized, cam_resized, use_rgb=True)
    
        # Create figure with multiple subplots
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
        # Original image
        axes[0].imshow(img_resized)
        axes[0].set_title('Original Image', fontsize=12)
        axes[0].axis('off')
    
        # GradCAM heatmap
        axes[1].imshow(cam_resized, cmap='jet')
        axes[1].set_title('GradCAM Heatmap', fontsize=12)
        axes[1].axis('off')
    
        # Overlay
        axes[2].imshow(visualization)
        axes[2].set_title(f'Overlay\nPredicted: {class_names[pred_class]}\nAge: {int(age)} years', fontsize=12)
        axes[2].axis('off')
    
        plt.tight_layout()
    
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
    
        return cam_resized, pred_class


# Initialize GradCAM with ConvNeXt Small's last convolutional layer
# For ConvNeXt Small, target the last stage (stage 3)
target_layer = model.backbone.stages[-1]  # Last stage of ConvNeXt

gradcam_explainer = ConvNeXtGradCAM(model, target_layer, cam_cls = GradCAMPlusPlus)

print("GradCAM explainer initialized successfully!")

# =====================================================================
# 2. BATCH GRAD-CAM VISUALIZATION
# =====================================================================
def visualize_batch_predictions(test_loader, model, class_names, num_samples=8, save_dir='/kaggle/working/gradcam_samples/'):
    """
    Visualize GradCAM++ for multiple test samples
    """
    os.makedirs(save_dir, exist_ok=True)
    
    model.eval()
    count = 0
    
    for images, labels, ages in test_loader:
        for i in range(min(num_samples - count, len(images))):
            img_tensor = images[i:i+1].to(device)
            age_tensor = ages[i:i+1].to(device)
            label = labels[i].item()
            age = ages[i].item()
            
            # Generate GradCAM
            cam, pred_class, _ = gradcam_explainer.generate_cam(img_tensor, age_tensor)
            
            # Prepare image
            img_np = img_tensor.cpu().squeeze().permute(1, 2, 0).numpy()
            # Denormalize
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img_np = std * img_np + mean
            img_np = np.clip(img_np, 0, 1)
            
            # Resize CAM to match image dimensions
            cam_resized = cv2.resize(cam, (img_np.shape[1], img_np.shape[0]))
            
            # Overlay
            visualization = show_cam_on_image(img_np, cam_resized, use_rgb=True)
            
            # Plot
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            
            axes[0].imshow(img_np)
            axes[0].set_title('Original', fontsize=12)
            axes[0].axis('off')
            
            axes[1].imshow(cam_resized, cmap='jet')
            axes[1].set_title('GradCAM', fontsize=12)
            axes[1].axis('off')
            
            axes[2].imshow(visualization)
            is_correct = "✓" if pred_class == label else "✗"
            # Denormalize age for display (multiply by 19)
            age_display = int(age * 19)
            axes[2].set_title(f'{is_correct} True: {class_names[label]}\nPred: {class_names[pred_class]}\nAge: {age_display} years', 
                            fontsize=12)
            axes[2].axis('off')
            
            plt.tight_layout()
            save_path = os.path.join(save_dir, f'sample_{count+1}.png')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
            
            count += 1
            if count >= num_samples:
                break
        
        if count >= num_samples:
            break

    print(f"Saved {count} GradCAM visualizations to {save_dir}")


# Generate visualizations for 8 test samples
visualize_batch_predictions(test_loader, model, class_names, num_samples=8


# =====================================================================
# FINAL SUMMARY
# =====================================================================

print("\n" + "="*80)

print("EXPLAINABILITY METHODS INITIALIZED AND EXECUTED")
print("="*80)
print("1. GradCAM - Visual attention maps for ConvNeXt Small")
print("2. Batch visualizations - Multiple sample analysis")


In [ ]:
# Example 1: Single image GradCAM
gradcam_explainer.visualize(
    image_path='/kaggle/working/ao_data/balanced_test/23r-M-2.1/0748_0422278263_01_WRI-L2_M013_0.png',
    age=13,
    class_names=class_names,
    save_path='/kaggle/working/gradcam_example.png'
)



In [ ]:

# ==========================
# AGE DISTRIBUTION ANALYSIS
# ==========================
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

def extract_age_from_filename(filename):
    """
    Extract age from filename pattern: _M013_ or _F025_ or -F019-, etc.
    Returns age as integer, or -1 if not found
    """
    match = re.search(r'[_\-][MFmf](\d{3})[_\-]', filename)
    if match:
        age = int(match.group(1))
        return age
    return -1

def collect_ages_from_directory(root_dir):
    """
    Collect all ages from images in directory structure
    Returns: list of ages, dict of {class_name: [ages]}
    """
    all_ages = []
    class_ages = {}
    
    # Iterate through class directories
    classes = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    
    for cls in classes:
        cls_dir = os.path.join(root_dir, cls)
        cls_ages = []
        
        for filename in os.listdir(cls_dir):
            if filename.endswith('.png') or filename.endswith('.jpg'):
                age = extract_age_from_filename(filename)
                if age >= 0:  # Valid age
                    all_ages.append(age)
                    cls_ages.append(age)
        
        class_ages[cls] = cls_ages
    
    return all_ages, class_ages

# ==========================
# COLLECT AGE DATA
# ==========================
out_train = "/kaggle/working/ao_data/balanced_train"
out_val = "/kaggle/working/ao_data/balanced_val"
out_test = "/kaggle/working/ao_data/balanced_test"

print("Collecting age data from datasets...")
train_ages, train_class_ages = collect_ages_from_directory(out_train)
val_ages, val_class_ages = collect_ages_from_directory(out_val)
test_ages, test_class_ages = collect_ages_from_directory(out_test)

print(f"\nCollected ages:")
print(f"  Train: {len(train_ages)} samples")
print(f"  Val: {len(val_ages)} samples")
print(f"  Test: {len(test_ages)} samples")

# ==========================
# AGE STATISTICS
# ==========================
def print_age_stats(ages, dataset_name):
    """Print age statistics"""
    ages_array = np.array(ages)
    print(f"\n{dataset_name} Age Statistics:")
    print(f"  Mean: {ages_array.mean():.2f} years")
    print(f"  Std: {ages_array.std():.2f} years")
    print(f"  Min: {ages_array.min()} years")
    print(f"  Max: {ages_array.max()} years")
    print(f"  Median: {np.median(ages_array):.2f} years")
    
    # Age distribution
    age_counts = Counter(ages)
    print(f"  Unique ages: {len(age_counts)}")
    print(f"  Most common ages: {age_counts.most_common(5)}")

print_age_stats(train_ages, "TRAIN")
print_age_stats(val_ages, "VAL")
print_age_stats(test_ages, "TEST")

# ==========================
# VISUALIZATION 1: Overall Age Distribution
# ==========================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Train
axes[0].hist(train_ages, bins=20, color='#3498db', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Age (years)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title(f'Train Set Age Distribution\n(n={len(train_ages)}, μ={np.mean(train_ages):.1f}, σ={np.std(train_ages):.1f})', 
                  fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 20)

# Val
axes[1].hist(val_ages, bins=20, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Age (years)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title(f'Validation Set Age Distribution\n(n={len(val_ages)}, μ={np.mean(val_ages):.1f}, σ={np.std(val_ages):.1f})', 
                  fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 20)

# Test
axes[2].hist(test_ages, bins=20, color='#2ecc71', alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Age (years)', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].set_title(f'Test Set Age Distribution\n(n={len(test_ages)}, μ={np.mean(test_ages):.1f}, σ={np.std(test_ages):.1f})', 
                  fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(0, 20)

plt.tight_layout()
plt.savefig('/kaggle/working/age_distribution_overall.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================
# VISUALIZATION 2: Combined Comparison
# ==========================
fig, ax = plt.subplots(figsize=(14, 6))

# Create bins
bins = np.arange(0, 21, 1)

# Plot overlapping histograms
ax.hist(train_ages, bins=bins, alpha=0.5, label=f'Train (n={len(train_ages)})', 
        color='#3498db', edgecolor='black')
ax.hist(val_ages, bins=bins, alpha=0.5, label=f'Val (n={len(val_ages)})', 
        color='#e74c3c', edgecolor='black')
ax.hist(test_ages, bins=bins, alpha=0.5, label=f'Test (n={len(test_ages)})', 
        color='#2ecc71', edgecolor='black')

ax.set_xlabel('Age (years)', fontsize=14)
ax.set_ylabel('Frequency', fontsize=14)
ax.set_title('Age Distribution Comparison: Train vs Val vs Test', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.set_xlim(0, 20)

plt.tight_layout()
plt.savefig('/kaggle/working/age_distribution_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================
# VISUALIZATION 3: Box Plot Comparison
# ==========================
fig, ax = plt.subplots(figsize=(10, 6))

data_to_plot = [train_ages, val_ages, test_ages]
labels = ['Train', 'Val', 'Test']
colors = ['#3498db', '#e74c3c', '#2ecc71']

bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True, 
                showmeans=True, meanline=True,
                medianprops=dict(color='red', linewidth=2),
                meanprops=dict(color='blue', linewidth=2, linestyle='--'))

# Color the boxes
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_ylabel('Age (years)', fontsize=14)
ax.set_title('Age Distribution Box Plot: Train vs Val vs Test', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 20)

# Add sample size annotations
for i, (ages, label) in enumerate(zip(data_to_plot, labels)):
    ax.text(i+1, -1, f'n={len(ages)}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/age_distribution_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================
# VISUALIZATION 4: Per-Class Age Distribution
# ==========================
# Get class names
class_names_sorted = sorted(train_class_ages.keys())
n_classes = len(class_names_sorted)

# Create subplots
fig, axes = plt.subplots(n_classes, 3, figsize=(18, 4*n_classes))

if n_classes == 1:
    axes = axes.reshape(1, -1)

for idx, cls in enumerate(class_names_sorted):
    train_cls_ages = train_class_ages.get(cls, [])
    val_cls_ages = val_class_ages.get(cls, [])
    test_cls_ages = test_class_ages.get(cls, [])
    
    # Train
    if len(train_cls_ages) > 0:
        axes[idx, 0].hist(train_cls_ages, bins=20, color='#3498db', alpha=0.7, edgecolor='black')
        axes[idx, 0].set_title(f'{cls} - Train (n={len(train_cls_ages)}, μ={np.mean(train_cls_ages):.1f})', 
                              fontsize=12, fontweight='bold')
    else:
        axes[idx, 0].text(0.5, 0.5, 'No data', ha='center', va='center', transform=axes[idx, 0].transAxes)
    axes[idx, 0].set_ylabel('Frequency', fontsize=10)
    axes[idx, 0].set_xlim(0, 20)
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Val
    if len(val_cls_ages) > 0:
        axes[idx, 1].hist(val_cls_ages, bins=20, color='#e74c3c', alpha=0.7, edgecolor='black')
        axes[idx, 1].set_title(f'{cls} - Val (n={len(val_cls_ages)}, μ={np.mean(val_cls_ages):.1f})', 
                              fontsize=12, fontweight='bold')
    else:
        axes[idx, 1].text(0.5, 0.5, 'No data', ha='center', va='center', transform=axes[idx, 1].transAxes)
    axes[idx, 1].set_xlim(0, 20)
    axes[idx, 1].grid(True, alpha=0.3)
    
    # Test
    if len(test_cls_ages) > 0:
        axes[idx, 2].hist(test_cls_ages, bins=20, color='#2ecc71', alpha=0.7, edgecolor='black')
        axes[idx, 2].set_title(f'{cls} - Test (n={len(test_cls_ages)}, μ={np.mean(test_cls_ages):.1f})', 
                              fontsize=12, fontweight='bold')
    else:
        axes[idx, 2].text(0.5, 0.5, 'No data', ha='center', va='center', transform=axes[idx, 2].transAxes)
    axes[idx, 2].set_xlim(0, 20)
    axes[idx, 2].grid(True, alpha=0.3)

# Add x-labels only to bottom row
for ax in axes[-1, :]:
    ax.set_xlabel('Age (years)', fontsize=12)

plt.tight_layout()
plt.savefig('/kaggle/working/age_distribution_per_class.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================
# SUMMARY TABLE
# ==========================
print("\n" + "="*80)
print("AGE DISTRIBUTION SUMMARY")
print("="*80)

summary_data = []
for cls in class_names_sorted:
    train_cls_ages = train_class_ages.get(cls, [])
    val_cls_ages = val_class_ages.get(cls, [])
    test_cls_ages = test_class_ages.get(cls, [])
    
    print(f"\n{cls}:")
    print(f"  Train: n={len(train_cls_ages):3d}, μ={np.mean(train_cls_ages) if len(train_cls_ages)>0 else 0:.1f}, σ={np.std(train_cls_ages) if len(train_cls_ages)>0 else 0:.1f}")
    print(f"  Val:   n={len(val_cls_ages):3d}, μ={np.mean(val_cls_ages) if len(val_cls_ages)>0 else 0:.1f}, σ={np.std(val_cls_ages) if len(val_cls_ages)>0 else 0:.1f}")
    print(f"  Test:  n={len(test_cls_ages):3d}, μ={np.mean(test_cls_ages) if len(test_cls_ages)>0 else 0:.1f}, σ={np.std(test_cls_ages) if len(test_cls_ages)>0 else 0:.1f}")

print("\n" + "="*80)
print("Age distribution analysis complete!")
print("Saved plots:")
print("   - /kaggle/working/age_distribution_overall.png")
print("   - /kaggle/working/age_distribution_comparison.png")
print("   - /kaggle/working/age_distribution_boxplot.png")
print("   - /kaggle/working/age_distribution_per_class.png")
print("="*80)
